# CE541E08 — Unit 3 · Day 23 — Conditions, Boolean Arrays and Quality Control

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 3 — The NumPy Library |
| **Session** | Day 23 of 45 |
| **CO** | CO3, CO4 |
| **Topics** | Boolean arrays · compound conditions · np.select · np.bincount · data QC · nan handling |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 23"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Boolean Arrays and Conditions

A **boolean array** is an array of True/False values — one per element — produced when you apply a comparison to a NumPy array. Today we extend this to:

- **Compound conditions** using `&` (AND) and `|` (OR) — for multi-threshold classification
- **`np.select`** — applying different values based on multiple conditions at once
- **`np.bincount`** — counting how many elements fall into each category
- **Quality control (QC)** — detecting bad sensor values and replacing them with `np.nan`

> **Important:** In NumPy, use `&` for element-wise AND and `|` for element-wise OR — **not** Python's `and`/`or` keywords, which only work on single values.

---
## Code Block 1 — Compound Boolean Conditions: Flood Warning System

### What this code does

We classify 30 days of streamflow into three zones — Normal, Warning, and Flood — using compound boolean conditions. We then extract the flood-day values and their day numbers.

### Why each step is taken

**`flow > 600` — flood boolean array:**
Compares every element to 600. Returns a True/False array of the same length. True means that day's flow exceeded the flood threshold.

**`(flow > 400) & (flow <= 600)` — warning zone:**
Two separate comparisons joined by `&` (element-wise AND). Each element must satisfy **both** conditions — greater than 400 AND at most 600 — to be True. This defines the warning zone between the two thresholds. The parentheses around each condition are required because `&` has higher precedence than `>` and `<=`.

**`np.where(is_flood)[0] + 1`:**
`np.where(condition)` returns a tuple. For a 1-D array, `[0]` gives the index array of True positions (0-based). Adding 1 converts to day numbers (1-based).

**`flow[is_flood]`:**
Boolean indexing — selects only the flow values where `is_flood` is True. Gives the actual discharge values on flood days.

**`flow[flow <= 600].mean()`:**
The complementary condition selects normal days. `.mean()` gives the baseline flow when no flood is occurring.

### Algorithm

```
1. Create 30-day streamflow array

2. is_flood   = flow > 600
   → True/False for each day

3. is_warning = (flow > 400) & (flow <= 600)
   → True only where BOTH conditions are met
   → parentheses required around each comparison

4. .sum() on boolean array → count of True values

5. np.where(is_flood)[0] + 1
   → indices of True values → day numbers (1-based)

6. flow[is_flood]   → actual flood discharge values
   flow[flow<=600].mean() → mean flow on normal days
```

### Expected output

```
Flood days   : 8
Warning days : 4
Flood flows  : [ 890 1245  987  756  678  890 1123  987]
Flood day nos: [ 4  5  6  7 19 20 21 22]
Normal mean  : 287.3 m3/s
```

In [ ]:
import numpy as np

# 30-day streamflow record (m³/s) — KRS station, July 2024
flow = np.array([234, 267, 312, 890, 1245, 987, 756, 543, 412, 345,
                 289, 245, 212, 198, 220, 265, 310, 456, 678, 890,
                 1123, 987, 765, 543, 421, 345, 289, 245, 212, 198])

# Single condition: True where flow exceeds flood threshold
is_flood = flow > 600

# Compound condition: True only where BOTH comparisons are True
# & is element-wise AND — must use parentheses around each comparison
is_warning = (flow > 400) & (flow <= 600)

# .sum() on a boolean array counts True values (True=1, False=0)
print(f"Flood days   : {is_flood.sum()}")
print(f"Warning days : {is_warning.sum()}")

# Boolean indexing: select only flood-day values
print(f"Flood flows  : {flow[is_flood]}")

# np.where(condition) returns a tuple; [0] gives the index array
# +1 converts from 0-based index to 1-based day number
print(f"Flood day nos: {np.where(is_flood)[0] + 1}")

# Complement condition: normal (non-flood) days
print(f"Normal mean  : {flow[flow <= 600].mean():.1f} m3/s")

### 🔁 Try this

Add a fourth category: **Extreme flood** where flow > 1000 m³/s.

- How many extreme flood days are there?
- What is the mean flow on extreme flood days?
- Use `(flow > 1000).sum()` and `flow[flow > 1000].mean()`

---
## Code Block 2 — np.select: IMD Rainfall Classification

### What this code does

We classify all 30 days of June 2024 rainfall into IMD categories simultaneously using `np.select` — then count the number of days in each category using `np.bincount`.

### Why each step is taken

**`np.select(conditions, choices, default)`:**
Takes a list of conditions and a matching list of output values. For each element, it applies the **first** condition that is True and returns the corresponding choice. If none match, it returns the default. This replaces a chain of `if-elif-elif-elif` applied to every element — all done in one vectorised call.

**Why the order of conditions matters:**
The conditions are checked in order and the first True one wins. Since `daily <= 15.5` is tested before `daily <= 64.4`, a value of 10 mm correctly gets category 1 (Light), not category 2 (Moderate). If the order were reversed, the less-specific condition would fire first.

**`np.bincount(cat)`:**
Counts how many times each integer value appears in the category array. For categories 0, 1, 2, 3, 4, 5 it returns a count for each. It is much faster than writing a loop to count occurrences.

**`.astype(int)` for the heavy flag:**
Converting a boolean array to integers gives 0 for False and 1 for True. This is useful for saving a QC flag column or computing weighted sums.

### Algorithm

```
1. Create 30-day daily rainfall array

2. np.select([cond1, cond2, cond3, ...],
             [val1,  val2,  val3,  ...],
             default=val_else)
   → applies first matching condition to each element
   → returns integer category code array

3. np.bincount(cat)
   → counts occurrences of each integer 0,1,2,...
   → one count per category

4. Convert to int flag: (daily >= 64.5).astype(int)
   → 1 where Heavy+, 0 otherwise
```

### Expected output

```
  No rain               :  14 days
  Light                 :   5 days
  Moderate              :   5 days
  Heavy                 :   4 days
  Very Heavy            :   2 days
Heavy+ flag: [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 1 0 0 0 0]
Heavy+ count: 6
```

In [ ]:
import numpy as np

# June 2024 daily rainfall (mm) — Cauvery basin
daily = np.array([0, 0, 12.4, 45.6, 0, 8.2, 23.1, 0, 0, 87.3,
                  34.5, 0, 56.2, 0, 18.9, 0, 0, 134.5, 22.3, 45.6,
                  0, 0, 67.8, 12.1, 0, 89.4, 33.2, 0, 45.1, 28.7])

# np.select: conditions tested in order — first True wins
# Categories: 0=No rain, 1=Light, 2=Moderate, 3=Heavy, 4=Very Heavy, 5=Ex.Heavy
cat = np.select(
    [daily == 0,       # condition 0: exactly zero
     daily <= 15.5,    # condition 1: 0.1 to 15.5 mm
     daily <= 64.4,    # condition 2: 15.6 to 64.4 mm
     daily <= 115.5,   # condition 3: 64.5 to 115.5 mm
     daily <= 204.4],  # condition 4: 115.6 to 204.4 mm
    [0, 1, 2, 3, 4],
    default=5          # Extremely Heavy: > 204.4 mm
)

names = ['No rain', 'Light', 'Moderate', 'Heavy', 'Very Heavy', 'Extremely Heavy']

# np.bincount: counts how many times each integer category appears
counts = np.bincount(cat)
for code, name in enumerate(names):
    if code < len(counts) and counts[code] > 0:
        print(f"  {name:<22}: {counts[code]:>3} days")

# Boolean to integer flag: True→1, False→0
# Used to create a binary QC flag column
heavy_flag = (daily >= 64.5).astype(int)
print(f"Heavy+ flag: {heavy_flag}")
print(f"Heavy+ count: {heavy_flag.sum()}")

### 🔁 Try this

Compute the **total rainfall in each IMD category** using boolean indexing.

For example, total rainfall on Heavy days: `daily[(daily >= 64.5) & (daily <= 115.5)].sum()`

- Which category contributed the most to the monthly total?
- What fraction of the total fell on Heavy+ days?

---
## Code Block 3 — Data Quality Control: Detecting and Replacing Bad Values

### What this code does

Real sensor data contains bad values — missing readings flagged as -999, physically impossible spikes, and negative values. We detect all bad values in one compound condition, replace them with `np.nan`, compute statistics on the valid data using `nan`-aware functions, and fill the gaps with the dataset mean.

### Why each step is taken

**`(flow_raw == -999) | (flow_raw > 5000) | (flow_raw < 0)` — compound OR:**
`|` is element-wise OR. An element is bad if it satisfies **any** of the three conditions: it is a missing-value sentinel (-999), a physically impossible spike (>5000 m³/s), or negative (physically impossible for streamflow). The result is a single True/False array marking all bad positions.

**`.astype(float)` before assigning nan:**
Integer arrays cannot hold `np.nan` (which is a floating-point concept). Converting to `float` first ensures the assignment works.

**`flow_clean[bad] = np.nan`:**
Boolean indexing on the left side of an assignment replaces selected elements. All bad positions are set to `np.nan` in one operation.

**`np.nanmean` and `np.nanmax`:**
The `nan`-aware versions of `mean` and `max` skip NaN values automatically. Regular `.mean()` would return NaN if any element is NaN.

**`np.where(np.isnan(flow_clean), np.nanmean(flow_clean), flow_clean)` — gap filling:**
Replaces each NaN with the dataset mean. `np.isnan` detects NaN positions, `np.where` selects the replacement value. This is the simplest gap-filling strategy — more sophisticated methods (linear interpolation, climatological mean) are used in practice.

### Algorithm

```
1. Create raw streamflow array with -999 (missing) and 9999 (spike)

2. Compound condition with |:
   bad = (flow == -999) | (flow > 5000) | (flow < 0)
   → True at every bad position

3. Convert to float (nan requires float dtype)
   flow_clean = flow_raw.astype(float)

4. Replace bad positions with NaN:
   flow_clean[bad] = np.nan

5. Statistics using nan-aware functions:
   np.nanmean(flow_clean) → mean of valid values only
   (~np.isnan(flow_clean)).sum() → count of valid values

6. Fill NaN gaps with the dataset mean:
   np.where(np.isnan(flow_clean), fill_value, flow_clean)
```

### Expected output

```
Bad values: 4 of 16
Valid mean  : 371.44 m3/s
Valid count : 12
Filled: [234.5 267.8 371.4 312.4 890.2 371.4 756.4 543.2
         371.4 345.6 289.4 245.1 371.4 198.7 212.3 178.9]
```

In [ ]:
import numpy as np

# Raw streamflow data with quality issues
# -999 = missing reading (sensor offline)
# 9999 = data spike (sensor malfunction)
flow_raw = np.array([234.5, 267.8, -999, 312.4, 890.2, -999, 756.4, 543.2,
                     9999, 345.6, 289.4, 245.1, -999, 198.7, 212.3, 178.9])

# Compound OR condition: bad if ANY of the three conditions is True
# | is element-wise OR (not Python's 'or' keyword)
bad = (flow_raw == -999) | (flow_raw > 5000) | (flow_raw < 0)
print(f"Bad values: {bad.sum()} of {len(flow_raw)}")

# Convert to float — np.nan is a float value, cannot store in int array
flow_clean = flow_raw.astype(float)

# Replace all bad positions with NaN using boolean indexing on the left side
flow_clean[bad] = np.nan

# nan-aware statistics — skip NaN values automatically
print(f"Valid mean  : {np.nanmean(flow_clean):.2f} m3/s")

# ~np.isnan gives True where NOT NaN (i.e. valid values)
print(f"Valid count : {(~np.isnan(flow_clean)).sum()}")

# Gap filling: replace NaN with dataset mean
fill_value = np.nanmean(flow_clean)
filled = np.where(np.isnan(flow_clean), fill_value, flow_clean)
print(f"Filled: {np.round(filled, 1)}")

### 🔁 Try this

Instead of filling with the mean, fill NaN gaps with the **median** using `np.nanmedian(flow_clean)`.

- How different is the result from the mean fill?
- Which is more appropriate when the data has outliers?

---
## Code Block 4 — SCS-CN Matrix Using a Loop

### What this code does

We compute a table of SCS-CN runoff depths for 10 storm sizes × 3 curve numbers. Each row is a storm depth, each column is a CN value. This produces the kind of pre-computed design chart used in stormwater engineering.

### Why each step is taken

**`S = 25400/CN_vals - 254` and `Ia = 0.2*S` — vectorised over CN:**
`CN_vals` is an array of 3 values. The formula applies to all three simultaneously, giving arrays `S` and `Ia` of length 3. This means the soil parameters for all three curve numbers are ready before the loop starts.

**Outer loop over storms, inner computation over CN:**
For each storm depth P, we check `P > Ia[i]` for each of the 3 CN values and apply the formula. The inner `for i in range(len(CN_vals))` steps through each column. This could be fully vectorised using broadcasting, but the loop version is easier to read and verify.

**Formatted print with `end=""`:**
Using `print(..., end="")` suppresses the newline so that columns for the same row appear on the same line. `print()` at the end of each storm row moves to the next line.

### Algorithm

```
1. Define storm depths: [15, 25, 35, ..., 105] mm (10 values)
   Define CN values: [65, 75, 85] (3 values)

2. Vectorised soil parameters:
   S  = 25400/CN_vals - 254   → array of 3 S values
   Ia = 0.2 * S               → array of 3 Ia values

3. Print table header

4. For each storm depth P:
   For each CN index i:
     If P > Ia[i]:
       Q = (P-Ia[i])² / (P-Ia[i]+S[i])
     Else:
       Q = 0.0
   Print one row: P, Q_CN65, Q_CN75, Q_CN85

5. Print closing separator
```

### Expected output

```
SCS-CN Runoff Matrix (mm)
   P(mm)  CN= 65  CN= 75  CN= 85
------------------------------
      15     0.0     0.0     0.5
      25     0.0     4.9    10.7
      35     0.6    12.1    22.5
      45     3.7    20.7    35.4
      55     9.3    30.4    49.2
      65    17.0    40.9    63.6
      75    26.4    52.0    78.3
      85    37.2    63.6    93.3
      95    49.2    75.6   108.5
     105    62.1    88.0   123.8
```

In [ ]:
import numpy as np

# Storm depths to analyse (mm)
storms  = np.array([15, 25, 35, 45, 55, 65, 75, 85, 95, 105])
# Curve numbers: 65=good pasture, 75=average cropland, 85=urban/impervious
CN_vals = np.array([65, 75, 85])

# Vectorised soil parameters — computed for all 3 CN values at once
S  = 25400 / CN_vals - 254   # potential retention (mm), shape (3,)
Ia = 0.2 * S                 # initial abstraction (mm), shape (3,)

print("SCS-CN Runoff Matrix (mm)")
# Header row
print(f"{'P(mm)':>8}", end="")
for CN in CN_vals:
    print(f"  CN={CN:>2}", end="")
print()
print("-" * 30)

# One row per storm depth
for P in storms:
    print(f"{P:>8}", end="")
    for i in range(len(CN_vals)):
        # SCS-CN formula: runoff only when P exceeds initial abstraction
        Q = (P - Ia[i])**2 / (P - Ia[i] + S[i]) if P > Ia[i] else 0.0
        print(f"  {Q:>5.1f}", end="")
    print()   # newline after each storm row

### 🔁 Try this

Add a fourth column for `CN = 95` (highly impervious — car park or roof).

- At what storm depth does runoff begin for CN=95?
- How does CN=95 compare to CN=65 at P=85 mm?

---
## Session Summary — Boolean Arrays and Quality Control

| Operation | Syntax | Notes |
|---|---|---|
| Single condition | `arr > 600` | Returns True/False array |
| Compound AND | `(arr > 400) & (arr <= 600)` | Both must be True; use `&` not `and` |
| Compound OR | `(arr == -999) \| (arr > 5000)` | Either True; use `\|` not `or` |
| Count True | `bool_arr.sum()` | True=1, False=0 |
| Select by condition | `arr[arr > x]` | Returns matching values |
| Get indices | `np.where(arr > x)[0]` | Returns index array |
| Multi-way classification | `np.select([c1,c2,...],[v1,v2,...],default)` | First matching condition wins |
| Count categories | `np.bincount(int_array)` | Count of each integer value |
| Replace bad with NaN | `arr[bad] = np.nan` | Array must be float dtype |
| Mean ignoring NaN | `np.nanmean(arr)` | Skips NaN automatically |
| Fill NaN | `np.where(np.isnan(arr), fill, arr)` | Replace NaN with a value |
| Boolean to int flag | `(arr > x).astype(int)` | 1=True, 0=False |

---
## Day 23 Assignment

```python
flow = np.array([234, 267, -999, 890, 1245, 987, 756, 543, 412, 345,
                 289, -999, 212, 198, 220, 265, 310, 99999, 678, 890])
```

1. Identify all bad values: `-999` (missing) and `> 5000` (spike). Count them.
2. Replace bad values with `np.nan`. Compute the valid count, mean, and max.
3. Create a flood flag array: `1` where flow > 600, `0` otherwise (after cleaning).

### ▶ Assignment cell

In [ ]:
import numpy as np

flow = np.array([234, 267, -999, 890, 1245, 987, 756, 543, 412, 345,
                 289, -999, 212, 198, 220, 265, 310, 99999, 678, 890])

# 1. Detect bad values
bad  = ???
print(f"Bad count   : {bad.sum()}")

# 2. Clean and compute statistics
flow_c = flow.astype(float)
flow_c[bad] = np.nan
print(f"Valid count : {(~np.isnan(flow_c)).sum()}")
print(f"Valid mean  : {np.nanmean(flow_c):.1f}")
print(f"Valid max   : {np.nanmax(flow_c):.1f}")

# 3. Flood flag (on cleaned data)
flood_flag = ???
print(f"Flood days  : {flood_flag.sum()}")

---
- [ ] Run all cells from top to bottom — verify outputs match expected outputs above
- [ ] Complete the assignment cell (replace `???` placeholders)
- [ ] Upload to GitHub: `Unit3_NumPy/CE541E08_U3_Day23.ipynb`
- [ ] Commit message: `Day 23 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*